In [10]:
"""
# Dataset sintético (España) - Apps de citas (2026)

Genera un dataset sintético (n=50.000) con distribuciones realistas y coherencia interna
por usuario (funnel consistente), diseñado para visualización.

## Calibración y fuentes de referencia
Las métricas estructurales de audiencia (usuarios únicos mensuales, tiempo de uso,
distribución por género y comunidad autónoma) se han calibrado a partir de estudios
públicos sobre consumo digital en España, correspondientes al período diciembre 2024,
publicados en el primer trimestre de 2025 por medidores oficiales del ecosistema digital
español (paneles de medición de audiencias digitales).

Métricas con respaldo en fuentes públicas contrastadas:
    - Usuarios únicos mensuales: ranking y volumen de las tres principales plataformas.
    - Duración mensual por usuario: media general ~211 min, hombres ~237 min, mujeres ~145 min.
    - Split de género: ~72,3% hombres / ~27,7% mujeres sobre 4,7M usuarios totales.
    - Tiempo mensual por CCAA: País Vasco, Baleares y Castilla-La Mancha superan
      la media nacional de forma consistente entre períodos.
    - Perfil de edad: usuarios de 25-34 años con mayor afinidad relativa,
      seguidos del segmento 18-24.

Nota para revisión externa: los valores de cuota de mercado de plataformas con menor
cobertura en estudios públicos (Bumble, Hinge, Meetic, Wapo, Boo, LOVOO, Jaumo)
son estimaciones proxy basadas en proporciones de mercado relativas. Deben tratarse
como supuestos de modelado, no como datos contrastados por fuente primaria.

## Principios de coherencia aplicados
- Las métricas mensuales se derivan de tasas por día activo × `dias_activos_mes`.
  Ejemplo: si un usuario presenta una tasa semanal de ~2 eventos, el total mensual dependerá
  de su número de días activos; se evita que el agregado mensual contradiga la frecuencia.
- `horas_mes` deriva de `minutos_mes`.
- `swipes_mes` deriva de `swipes_diarios` × `dias_activos_mes` (no de 30 días fijos).
- `matches_mes` está acotado por un potencial de interacciones plausible (no excede límites lógicos).
- Conversaciones y respuestas dependen de `matches_mes` (no se generan conversaciones sin matches).
- Las variables booleanas se expresan como texto ("Sí"/"No") para facilitar su uso en herramientas BI.

## Métricas derivadas incluidas
El dataset incluye métricas calculadas organizadas por tramo del funnel:

    Actividad:
        minutos_por_sesion      → minutos_mes / (sesiones_por_dia × dias_activos_mes)

    Visibilidad:
        ratio_likes_enviados_recibidos → likes_enviados_diarios / likes_recibidos_diarios
                                         (>1 = usuario más activo que visible)

    Funnel superior (swipe → match):
        calidad_match           → likes_enviados_mes / matches_mes (más precisa que tasa_exito_match)
        tasa_exito_match        → matches_mes / swipes_mes (métrica conservadora)
        matches_por_hora        → matches_mes / horas_mes

    Funnel medio (match → conversación):
        conversion_conversacion → conversaciones_enviadas_mes / matches_mes

    Calidad conversacional:
        tasa_respuesta          → respuestas_recibidas_mes / conversaciones_enviadas_mes
        indice_reciprocidad     → calidad_match × tasa_respuesta (índice compuesto)

    Funnel completo (swipe → cita):
        eficiencia_funnel       → cita / swipes_mes (valor casi siempre ~0, efecto visual potente)
        tasa_conversion_cita    → cita / matches_mes (último tramo del funnel)

    Segmentación conductual:
        perfil_usuario          → Eficiente | Selectivo | Conversador | Explorador | Pasivo

## El dataset es 100% sintético y no contiene datos reales de usuarios.
"""

import pandas as pd
import numpy as np

# ============================================================
# PARÁMETROS
# ============================================================
np.random.seed(42)
n = 50_000

# Dispersión relativa del tiempo mensual por app (variabilidad entre usuarios)
TIME_CV = 0.35

# CCAA con mayor dedicación media según estudios públicos de audiencia digital (dic 2024)
# Fuente: tiempo mensual por usuario por comunidad autónoma
TOP_TIME_REGIONS = ["País Vasco", "Baleares", "Castilla-La Mancha"]
REGION_TIME_BOOST = 1.15

# ============================================================
# 1) BASE: ID + DEMOGRAFÍA + CONTEXTO
# ============================================================
df = pd.DataFrame({"id_usuario": np.arange(1, n + 1)})

# Género — split calibrado contra estudios públicos de audiencia digital dic 2024
# 3,4M hombres / 1,3M mujeres sobre 4,7M totales → 72,3% / 27,7%
df["genero"] = np.random.choice(["Hombre", "Mujer"], size=n, p=[0.73, 0.27])

# CCAA (aproximación por pesos poblacionales; se normaliza por seguridad)
ccaa = [
    "Andalucía", "Aragón", "Asturias", "Baleares", "Canarias", "Cantabria",
    "Castilla-La Mancha", "Castilla y León", "Cataluña", "Comunidad Valenciana",
    "Extremadura", "Galicia", "Madrid", "Murcia", "Navarra", "País Vasco", "La Rioja"
]
ccaa_weights = np.array([
    0.18, 0.03, 0.02, 0.03, 0.05, 0.01, 0.04, 0.05, 0.16, 0.11,
    0.02, 0.06, 0.14, 0.03, 0.01, 0.05, 0.01
], dtype=float)
df["ccaa"] = np.random.choice(ccaa, size=n, p=ccaa_weights / ccaa_weights.sum())

# ============================================================
# 2) APP PRINCIPAL (10 apps) + EDAD DEPENDIENTE DE APP
# ============================================================

# Usuarios únicos mensuales — calibrado contra estudios públicos de audiencia digital, diciembre 2024
# Tinder: 1.500.000 | Badoo: 776.000 | Grindr: 635.600 (contrastados en fuentes públicas)
# Resto: estimaciones proxy sin fuente primaria verificable
app_uniques = {
    "Tinder":  1_500_000,
    "Badoo":     776_000,
    "Grindr":    635_600,
    "Bumble":    491_300,
    "Hinge":     314_300,
    "Meetic":    307_000,
    "Wapo":      229_800,
    "Boo":       215_500,
    "LOVOO":     152_600,
    "Jaumo":     117_700,
}
apps = list(app_uniques.keys())
base_probs = np.array([app_uniques[a] for a in apps], dtype=float)
base_probs = base_probs / base_probs.sum()

# Ajuste por género (introduce diferencias de preferencia sin romper el mercado global)
male_boost   = {"Grindr": 1.35, "Wapo": 1.25, "Tinder": 0.95, "Badoo": 0.95}
female_boost = {"Tinder": 1.10, "Badoo": 1.05, "Grindr": 0.55, "Wapo":  0.65}

def _adjust_probs(apps_list, base_p, boosts):
    p = base_p.copy()
    idx = {a: i for i, a in enumerate(apps_list)}
    for a, f in boosts.items():
        if a in idx:
            p[idx[a]] *= f
    return p / p.sum()

p_h = _adjust_probs(apps, base_probs, male_boost)
p_m = _adjust_probs(apps, base_probs, female_boost)

df["app_principal"] = None
mask_h = df["genero"] == "Hombre"
mask_m = ~mask_h

df.loc[mask_h, "app_principal"] = np.random.choice(apps, size=int(mask_h.sum()), p=p_h)
df.loc[mask_m, "app_principal"] = np.random.choice(apps, size=int(mask_m.sum()), p=p_m)

# Edad: distribución sesgada a jóvenes (Beta) con desplazamientos por app
# Millennials (25-34) mayor afinidad; Gen Z segundo grupo — estudios públicos 2025
age_min, age_max = 18, 64
u = np.random.beta(2, 6, size=n)
age_base = age_min + u * (age_max - age_min)

app_shift = df["app_principal"].map({
    "Tinder": -2.0, "Bumble": -1.5, "Hinge": -1.0,
    "Grindr":  -0.5, "Wapo":  -0.5, "Badoo": +1.5,
    "Meetic":  +7.0, "Boo":   -0.5, "LOVOO": +0.5, "Jaumo": +1.0
}).astype(float).values

df["edad"] = np.round(age_base + app_shift + np.random.normal(0, 2.5, size=n)).astype(int)
df["edad"] = np.clip(df["edad"], age_min, age_max)

# ============================================================
# 3) SOCIODEMOGRAFÍA
# ============================================================
edu_levels   = ["Primarios", "Secundario", "Universitario", "Postgrado"]
edu_weights  = np.array([0.194, 0.35, 0.35, 0.106], dtype=float)
df["nivel_estudios"] = np.random.choice(edu_levels, size=n, p=edu_weights / edu_weights.sum())

income_levels  = ["Bajo", "Medio", "Medio-Alto", "Alto"]
income_weights = np.array([0.25, 0.50, 0.16, 0.09], dtype=float)
df["nivel_ingresos"] = np.random.choice(income_levels, size=n, p=income_weights / income_weights.sum())

# Orientación sexual (dependiente de app)
df["orientacion_sexual"] = None
mask_lgbt = df["app_principal"].isin(["Grindr", "Wapo"])
n_lgbt = int(mask_lgbt.sum())
n_non  = int((~mask_lgbt).sum())

df.loc[mask_lgbt,  "orientacion_sexual"] = np.random.choice(
    ["Gay", "Bi", "LGBTQ+"], size=n_lgbt, p=[0.60, 0.25, 0.15]
)
df.loc[~mask_lgbt, "orientacion_sexual"] = np.random.choice(
    ["Heterosexual", "Bi"], size=n_non, p=[0.90, 0.10]
)

# ============================================================
# 4) COMPORTAMIENTO (COHERENTE POR INDIVIDUO)
# ============================================================

# Factor latente (heterogeneidad individual)
# Representa diferencias no observadas (p.ej. calidad de perfil) para generar correlaciones realistas.
score_atractivo = np.random.normal(0, 1, size=n)

# 4.1 Tiempo mensual por app (minutos) + variabilidad + ajustes
# Fuente confirmada GfK DAM dic 2024: Grindr = 612 min (10h 12m)
# Resto: estimaciones proxy coherentes con media general de 211 min
app_time_mean_min = {
    "Tinder":  157.7,
    "Badoo":   169.6,
    "Grindr":  612.0,   # 10h 12m — calibrado contra estudios públicos dic 2024
    "Bumble":   95.7,
    "Hinge":    65.6,
    "Meetic":   77.9,
    "Wapo":    192.0,
    "Boo":      68.7,
    "LOVOO":    47.8,
    "Jaumo":   123.9,
}
base_time = df["app_principal"].map(app_time_mean_min).astype(float).values

time_noise  = np.random.normal(0, TIME_CV, size=n)
minutos_mes = base_time * (1 + time_noise) * (1 + 0.05 * score_atractivo)

# Ajuste por género — hombres ~237 min / mujeres ~145 min (estudios públicos dic 2024)
# Ratio real: 1.63×; se aproxima con ×1.10 / ×0.90 sobre las bases por app
minutos_mes *= np.where(df["genero"].values == "Hombre", 1.10, 0.90)

# Boost para CCAA líderes en tiempo mensual (estudios públicos dic 2024)
minutos_mes *= np.where(df["ccaa"].isin(TOP_TIME_REGIONS).values, REGION_TIME_BOOST, 1.0)

df["minutos_mes"]  = np.clip(minutos_mes, 15, 1200).round().astype(int)
df["horas_mes"]    = (df["minutos_mes"] / 60.0).round(2)

# 4.2 Días activos al mes (coherentes con tiempo mensual)
base_days = 4 + (df["minutos_mes"] / 60.0) * 1.2
df["dias_activos_mes"] = np.clip(
    np.round(base_days + np.random.normal(0, 3, size=n)), 1, 30
).astype(int)

# 4.3 Sesiones por día (colas realistas con Gamma)
sessions  = np.random.gamma(shape=2, scale=1.2, size=n)
sessions *= (1 + 0.02 * np.clip(df["dias_activos_mes"] - 10, 0, 20))
df["sesiones_por_dia"] = np.clip(np.round(sessions, 1), 0.5, 15)

# 4.4 Frecuencia semanal (derivada de días activos)
freq_week_cont = (df["dias_activos_mes"] / 4.0).clip(0.5, 14)
bins = np.array([2, 3, 4, 5, 7, 14])
df["frecuencia_semanal"] = bins[
    np.argmin(np.abs(freq_week_cont.values[:, None] - bins[None, :]), axis=1)
]

# 4.5 Swipes diarios (coherentes con minutos por día activo)
min_por_dia_activo = (df["minutos_mes"] / df["dias_activos_mes"]).clip(5, 240)

lambda_swipes = 15 + 0.45 * min_por_dia_activo + 6 * np.clip(score_atractivo, -1, 2)
lambda_swipes = np.clip(lambda_swipes, 5, 250)

df["swipes_diarios"] = np.random.poisson(lam=lambda_swipes).clip(1, 400).astype(int)

# 4.6 Likes enviados diarios (subconjunto de swipes diarios)
p_like = 0.55 - 0.05 * np.clip(score_atractivo, -2, 2)
p_like = np.clip(p_like, 0.25, 0.80)

df["likes_enviados_diarios"] = np.random.binomial(
    df["swipes_diarios"], p_like
).clip(0, 400).astype(int)

# 4.7 Likes recibidos al mes (tasa por día activo × días activos)
base_like_recv_day = np.where(df["genero"].values == "Mujer", 6.0, 0.8)

app_recv_mult = df["app_principal"].map({
    "Tinder": 1.10, "Badoo": 1.00, "Grindr": 0.95, "Bumble": 1.05, "Hinge": 0.90,
    "Meetic": 0.85, "Wapo":  0.95, "Boo":    0.90, "LOVOO":  0.90, "Jaumo": 0.95
}).astype(float).values

latent_mult = np.clip(1 + 0.35 * score_atractivo, 0.35, 2.2)

likes_recibidos_dia = base_like_recv_day * app_recv_mult * latent_mult
df["likes_recibidos_mes"] = np.random.poisson(
    lam=likes_recibidos_dia * df["dias_activos_mes"]
).astype(int)
df["likes_recibidos_mes"] = np.clip(df["likes_recibidos_mes"], 0, 3000).astype(int)

# 4.8 Matches al mes (acotados por potencial)
likes_enviados_mes = (df["likes_enviados_diarios"] * df["dias_activos_mes"]).astype(int)

base_match_rate = np.where(df["genero"].values == "Mujer", 0.12, 0.06)
app_match_mult  = df["app_principal"].map({
    "Tinder": 1.00, "Badoo": 0.90, "Grindr": 1.10, "Bumble": 1.05, "Hinge": 1.00,
    "Meetic": 0.85, "Wapo":  1.00, "Boo":    0.95, "LOVOO":  0.85, "Jaumo": 0.90
}).astype(float).values

match_p = base_match_rate * app_match_mult * np.clip(1 + 0.15 * score_atractivo, 0.6, 1.6)
match_p = np.clip(match_p, 0.01, 0.30)

potential = np.minimum(df["likes_recibidos_mes"].values, likes_enviados_mes.values)
potential = np.clip(potential, 0, 500).astype(int)

df["matches_mes"] = np.random.binomial(n=potential, p=match_p).astype(int)
df["matches_mes"] = np.clip(df["matches_mes"], 0, 500).astype(int)

# 4.9 Respuestas y conversaciones (dependientes de matches)
df["respuestas_recibidas_mes"]   = np.random.poisson(
    lam=df["matches_mes"] * 0.60
).clip(0, 600).astype(int)
df["conversaciones_enviadas_mes"] = np.random.binomial(
    n=df["matches_mes"], p=0.65
).clip(0, 600).astype(int)

# 4.10 Tasa de respuesta (proxy de reciprocidad)
df["tasa_respuesta"] = np.where(
    df["conversaciones_enviadas_mes"] > 0,
    (df["respuestas_recibidas_mes"] / df["conversaciones_enviadas_mes"]).clip(0, 1),
    0.0
).round(3)

# 4.11 Flags Sí/No coherentes con el funnel
p_envio = np.clip(0.30 + 0.01 * df["conversaciones_enviadas_mes"], 0.05, 0.95)
df["envio_mensaje"] = np.where(np.random.rand(n) < p_envio, "Sí", "No")
df.loc[df["matches_mes"] == 0, "envio_mensaje"] = "No"

p_ghost = (
    0.18
    + 0.15 * (df["matches_mes"] > 10).astype(int)
    + 0.20 * (df["envio_mensaje"] == "No").astype(int)
)
p_ghost = np.clip(p_ghost, 0.05, 0.75)
df["hizo_ghosting"] = np.where(np.random.rand(n) < p_ghost, "Sí", "No")
df.loc[df["matches_mes"] == 0, "hizo_ghosting"] = "No"

p_cita = (
    0.02
    + 0.01 * np.clip(df["matches_mes"], 0, 30)
    + 0.08 * (df["envio_mensaje"] == "Sí").astype(int)
)
p_cita = np.clip(p_cita, 0.00, 0.60)
df["realizo_cita"] = np.where(np.random.rand(n) < p_cita, "Sí", "No")
df.loc[df["matches_mes"] == 0, "realizo_cita"] = "No"

# ============================================================
# 5) MÉTRICAS DERIVADAS — FUNNEL COMPLETO
# ============================================================

# --- 5.1 Volúmenes agregados mensuales ---

df["swipes_mes"] = (
    df["swipes_diarios"] * df["dias_activos_mes"]
).clip(0, 20_000).astype(int)

likes_enviados_mes_final = (
    df["likes_enviados_diarios"] * df["dias_activos_mes"]
).clip(0, 20_000).astype(int)
df["likes_enviados_mes"] = likes_enviados_mes_final

# --- 5.2 Intensidad de sesión ---
# Duración media por sesión individual (minutos)
# Permite comparar calidad de uso frente a cantidad de días activos
df["minutos_por_sesion"] = np.where(
    (df["sesiones_por_dia"] > 0) & (df["dias_activos_mes"] > 0),
    (df["minutos_mes"] / (df["sesiones_por_dia"] * df["dias_activos_mes"])).clip(1, 120),
    0.0
).round(1)

# --- 5.3 Métricas de visibilidad: ratio likes enviados / recibidos ---
# Valores >1 indican usuarios más activos que atractivos en términos de recepción
# Marcadamente diferente por género; útil para heatmap género × app
likes_recibidos_mes_dia = (df["likes_recibidos_mes"] / df["dias_activos_mes"].clip(1)).clip(0)
likes_enviados_dia = df["likes_enviados_diarios"].astype(float)

df["ratio_likes_enviados_recibidos"] = np.where(
    likes_recibidos_mes_dia > 0,
    (likes_enviados_dia / likes_recibidos_mes_dia).clip(0, 50),
    np.where(likes_enviados_dia > 0, 50.0, 0.0)
).round(2)

# --- 5.4 Eficiencia del funnel superior: likes → matches ---
# Más precisa que tasa_exito_match (swipes incluyen rechazos explícitos)
df["calidad_match"] = np.where(
    df["likes_enviados_mes"] > 0,
    (df["matches_mes"] / df["likes_enviados_mes"]).clip(0, 1),
    0.0
).round(4)

# Tasa de éxito sobre swipes totales (métrica más conservadora)
df["tasa_exito_match"] = np.where(
    df["swipes_mes"] > 0,
    (df["matches_mes"] / df["swipes_mes"]).clip(0, 1),
    0.0
).round(4)

# Rendimiento por hora invertida
df["matches_por_hora"] = np.where(
    df["horas_mes"] > 0,
    (df["matches_mes"] / df["horas_mes"]),
    0.0
).round(3)

# --- 5.5 Eficiencia del funnel medio: matches → conversación ---
# Qué proporción de matches se convierte en intento real de contacto
df["conversion_conversacion"] = np.where(
    df["matches_mes"] > 0,
    (df["conversaciones_enviadas_mes"] / df["matches_mes"]).clip(0, 1),
    0.0
).round(4)

# --- 5.6 Reciprocidad y calidad conversacional ---
# tasa_respuesta ya calculada en sección 4.10 (respuestas / conversaciones enviadas)

# Índice de reciprocidad compuesto: combina éxito de match con éxito conversacional
# Captura usuarios que no solo obtienen matches sino que generan interacción real
df["indice_reciprocidad"] = (
    df["tasa_respuesta"] * df["calidad_match"]
).round(5)

# --- 5.7 Eficiencia del funnel completo: swipes → cita ---
# La métrica más dramática visualmente; el denominador (swipes) hace que el valor
# sea casi siempre muy pequeño, lo que ilustra el funnel real de las apps de citas
df["eficiencia_funnel"] = np.where(
    df["swipes_mes"] > 0,
    ((df["realizo_cita"] == "Sí").astype(int) / df["swipes_mes"]).clip(0, 1),
    0.0
).round(6)

# Tasa de conversión en el último tramo del funnel: matches → cita
df["tasa_conversion_cita"] = np.where(
    df["matches_mes"] > 0,
    ((df["realizo_cita"] == "Sí").astype(int) / df["matches_mes"]).clip(0, 1),
    0.0
).round(4)

# ============================================================
# 6) SEGMENTACIÓN DE PERFIL DE USUARIO
# ============================================================
# Clasificación conductual derivada del comportamiento en el funnel.
# Permite filtros narrativos en dashboard sin necesidad de gráficos complejos.
#
# Lógica de asignación (por orden de prioridad):
#   Eficiente   → pocos swipes pero alta conversión a cita
#   Selectivo   → pocos swipes, alta calidad de match (tasa_exito_match alta)
#   Conversador → muchos matches, muchas conversaciones, pocas citas
#   Explorador  → muchos swipes, pocos matches (tasa_exito_match baja)
#   Pasivo      → baja actividad general (swipes_mes bajo, sin citas)

swipes_med    = float(np.median(df["swipes_mes"]))
match_med     = float(np.median(df["tasa_exito_match"]))
conv_cita_med = float(np.median(df["tasa_conversion_cita"][df["tasa_conversion_cita"] > 0]))

conditions = [
    # Eficiente: actividad moderada-baja pero llega a cita
    (df["swipes_mes"] <= swipes_med) & (df["realizo_cita"] == "Sí"),
    # Selectivo: pocos swipes y match rate alto (por encima de la mediana)
    (df["swipes_mes"] <= swipes_med) & (df["tasa_exito_match"] >= match_med),
    # Conversador: muchos matches y conversaciones pero raramente cita
    (df["matches_mes"] > df["matches_mes"].median()) &
    (df["conversaciones_enviadas_mes"] > df["conversaciones_enviadas_mes"].median()) &
    (df["realizo_cita"] == "No"),
    # Explorador: muchos swipes, match rate por debajo de la mediana
    (df["swipes_mes"] > swipes_med) & (df["tasa_exito_match"] < match_med),
]
choices = ["Eficiente", "Selectivo", "Conversador", "Explorador"]

df["perfil_usuario"] = np.select(conditions, choices, default="Pasivo")

# ============================================================
# 7) VARIABLE TÉCNICA (OPCIONAL)
# ============================================================
df["score_atractivo"] = np.round(score_atractivo, 2)

# ============================================================
# 8) ORDEN FINAL DE COLUMNAS
# Lógica de lectura: identidad → demografía → contexto app →
# tiempo y actividad → funnel de interacción (volúmenes) →
# métricas de eficiencia (ratios) → comportamiento cualitativo →
# segmentación → variable técnica
# ============================================================
ordered_cols = [
    # Identidad
    "id_usuario",
    # Demografía
    "genero", "edad", "ccaa",
    # Contexto app y perfil socioeconómico
    "app_principal", "nivel_estudios", "nivel_ingresos", "orientacion_sexual",
    # Tiempo y actividad
    "minutos_mes", "horas_mes", "dias_activos_mes",
    "sesiones_por_dia", "minutos_por_sesion", "frecuencia_semanal",
    # Funnel — volúmenes (swipe → like → match → conversación → cita)
    "swipes_diarios", "swipes_mes",
    "likes_enviados_diarios", "likes_enviados_mes", "likes_recibidos_mes",
    "ratio_likes_enviados_recibidos",
    "matches_mes",
    "conversaciones_enviadas_mes", "respuestas_recibidas_mes",
    # Métricas de eficiencia por tramo de funnel
    "calidad_match",           # likes enviados → matches
    "tasa_exito_match",        # swipes → matches (métrica conservadora)
    "matches_por_hora",        # rendimiento por tiempo invertido
    "conversion_conversacion", # matches → conversaciones
    "tasa_respuesta",          # conversaciones → respuestas recibidas
    "indice_reciprocidad",     # calidad_match × tasa_respuesta (índice compuesto)
    "tasa_conversion_cita",    # matches → cita
    "eficiencia_funnel",       # swipes → cita (funnel completo)
    # Comportamiento cualitativo
    "envio_mensaje", "hizo_ghosting", "realizo_cita",
    # Segmentación conductual
    "perfil_usuario",
    # Variable técnica
    "score_atractivo",
]
df = df[ordered_cols]

# ============================================================
# 8) EXPORT + CHEQUEOS DE COHERENCIA
# ============================================================
out_path = "dating_apps_espana_completo_2026.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print("CSV guardado en:", out_path)

bad_conv  = (df["matches_mes"] == 0) & (df["conversaciones_enviadas_mes"] > 0)
print("Incoherencias (matches=0 pero conversaciones_enviadas>0):", int(bad_conv.sum()))

bad_date  = (df["matches_mes"] == 0) & (df["realizo_cita"] == "Sí")
print("Incoherencias (matches=0 pero cita=Sí):", int(bad_date.sum()))

bad_swipes = df["swipes_mes"] != (df["swipes_diarios"] * df["dias_activos_mes"])
print("Incoherencias (swipes_mes no cuadra):", int(bad_swipes.sum()))

bad_likes  = df["likes_enviados_mes"] != (df["likes_enviados_diarios"] * df["dias_activos_mes"]).clip(0, 20_000)
print("Incoherencias (likes_enviados_mes no cuadra):", int(bad_likes.sum()))

print(f"\nPerfiles de usuario:\n{df['perfil_usuario'].value_counts().to_string()}")
print("\nEjemplo de filas:")
print(df.head())

CSV guardado en: dating_apps_espana_completo_2026.csv
Incoherencias (matches=0 pero conversaciones_enviadas>0): 0
Incoherencias (matches=0 pero cita=Sí): 0
Incoherencias (swipes_mes no cuadra): 0
Incoherencias (likes_enviados_mes no cuadra): 0

Perfiles de usuario:
perfil_usuario
Selectivo      24472
Pasivo         14238
Conversador    10573
Eficiente        717

Ejemplo de filas:
   id_usuario  genero  edad      ccaa app_principal nivel_estudios  \
0           1  Hombre    31    Madrid        Grindr  Universitario   
1           2   Mujer    49  Cataluña        Grindr  Universitario   
2           3   Mujer    31    Aragón        Tinder      Primarios   
3           4  Hombre    48   Galicia        Grindr     Secundario   
4           5  Hombre    27  Cataluña         Badoo     Secundario   

  nivel_ingresos orientacion_sexual  minutos_mes  horas_mes  ...  \
0          Medio                Gay          501       8.35  ...   
1          Medio             LGBTQ+          689      11.48